# 🤖 Notebook 02 — Fine-Tuning Task A (OFF/NOT)

**Hierarchical Offensive Language Detection: BERT vs DeBERTa**  

---

## Tujuan
Melatih BERT dan DeBERTa untuk **Task A**: mengklasifikasikan tweet sebagai:
- `OFF` = Offensive (ofensif)
- `NOT` = Not Offensive (tidak ofensif)

## Alur Notebook
```
STEP 1  → Install & Import
STEP 2  → Konfigurasi (pilih model di sini)
STEP 3  → Load & Split Dataset
STEP 4  → Preprocessing Teks
STEP 5  → Tokenisasi & Dataset PyTorch
STEP 6  → Load Model
STEP 7  → Hitung Class Weights
STEP 8  → Training Loop
STEP 9  → Evaluasi Test Set
STEP 10 → Simpan Hasil
```

> ⚙️ **Cara pakai:** Ubah variabel di STEP 2 untuk ganti model atau seed, lalu jalankan semua cell dari atas.

---
## STEP 1 — Install & Import Library

**Library yang dibutuhkan:**
- `transformers` → HuggingFace: untuk load model BERT/DeBERTa
- `torch` → PyTorch: framework deep learning
- `sklearn` → untuk hitung F1, precision, recall
- `pandas`, `numpy` → untuk mengolah data

In [3]:
# Install jika belum ada
# !pip install transformers torch scikit-learn pandas numpy statsmodels

import os, re, json, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score,
    accuracy_score, confusion_matrix
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from torch.optim import AdamW
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Cek GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  GPU tidak tersedia, training akan lambat di CPU')
    print('   Disarankan: Google Colab dengan GPU T4')

print('\n✅ Semua library berhasil diimport!')

Device: cpu
⚠️  GPU tidak tersedia, training akan lambat di CPU
   Disarankan: Google Colab dengan GPU T4

✅ Semua library berhasil diimport!


---
## STEP 2 — Konfigurasi

> **⚙️ INI YANG DIUBAH-UBAH untuk setiap run!**
>
> Jalankan notebook ini **6x** untuk Task A:
> - Run 1: BERT  + weighted,    seed=42
> - Run 2: BERT  + weighted,    seed=123
> - Run 3: BERT  + weighted,    seed=999
> - Run 4: DeBERTa + weighted,  seed=42
> - Run 5: DeBERTa + weighted,  seed=123
> - Run 6: DeBERTa + weighted,  seed=999
> 
> (+ 6 run lagi dengan `USE_WEIGHTED_LOSS = False` untuk ablasi)

In [4]:
# ═══════════════════════════════════════════════
# UBAH BAGIAN INI UNTUK SETIAP RUN
# ═══════════════════════════════════════════════

MODEL_CHOICE = 'bert'       # ← Ganti: 'bert' atau 'deberta'
SEED         = 42           # ← Ganti: 42, 123, atau 999
USE_WEIGHTED = True         # ← Ganti: True (weighted) atau False (ablasi)

# ═══════════════════════════════════════════════
# Hyperparameter (tidak perlu diubah)
# ═══════════════════════════════════════════════
EPOCHS      = 5
BATCH_SIZE  = 32
LR          = 2e-5
MAX_LENGTH  = 128
VAL_RATIO   = 0.1

# ═══════════════════════════════════════════════
# Path
# ═══════════════════════════════════════════════
BASE_DIR    = os.path.abspath('..')   # satu folder di atas /notebook
DATA_DIR    = os.path.join(BASE_DIR, 'dataset')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ═══════════════════════════════════════════════
# Mapping model name
# ═══════════════════════════════════════════════
MODEL_MAP = {
    'bert'   : 'bert-base-uncased',
    'deberta': 'microsoft/deberta-v3-base',
}
MODEL_NAME = MODEL_MAP[MODEL_CHOICE]

# Label
LABEL2ID = {'NOT': 0, 'OFF': 1}
ID2LABEL = {0: 'NOT', 1: 'OFF'}

# Nama run (untuk menyimpan hasil)
WL_TAG   = 'wl' if USE_WEIGHTED else 'nowl'
RUN_NAME = f'{MODEL_CHOICE}_seed{SEED}_{WL_TAG}'

print('═' * 45)
print('KONFIGURASI RUN SAAT INI')
print('═' * 45)
print(f'  Model          : {MODEL_NAME}')
print(f'  Random Seed    : {SEED}')
print(f'  Weighted Loss  : {USE_WEIGHTED}')
print(f'  Epochs         : {EPOCHS}')
print(f'  Batch Size     : {BATCH_SIZE}')
print(f'  Learning Rate  : {LR}')
print(f'  Max Length     : {MAX_LENGTH}')
print(f'  Run Name       : {RUN_NAME}')
print('═' * 45)

═════════════════════════════════════════════
KONFIGURASI RUN SAAT INI
═════════════════════════════════════════════
  Model          : bert-base-uncased
  Random Seed    : 42
  Weighted Loss  : True
  Epochs         : 5
  Batch Size     : 32
  Learning Rate  : 2e-05
  Max Length     : 128
  Run Name       : bert_seed42_wl
═════════════════════════════════════════════


---
## STEP 3 — Load & Split Dataset

Dataset dibagi menjadi:
- **Training set** (90%): untuk melatih model
- **Validation set** (10%): untuk memilih model terbaik saat training
- **Test set**: untuk evaluasi akhir (data resmi OLID, tidak boleh dipakai saat training)

In [5]:
def set_seed(seed):
    """Agar hasil bisa direproduksi dengan seed yang sama."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

# ── Load training ──
df_full = pd.read_csv(os.path.join(DATA_DIR, 'olid-training-v1.0.tsv'), sep='\t')
df_full = df_full.dropna(subset=['tweet', 'subtask_a'])
df_full['label'] = df_full['subtask_a'].map(LABEL2ID)
df_full = df_full.dropna(subset=['label'])
df_full['label'] = df_full['label'].astype(int)

# ── Train / Val split (stratified) ──
# Stratified = proporsi OFF/NOT tetap sama di train dan val
df_train, df_val = train_test_split(
    df_full[['tweet', 'label']],
    test_size=VAL_RATIO,
    random_state=SEED,
    stratify=df_full['label']
)

# ── Load test set ──
test_raw = pd.read_csv(os.path.join(DATA_DIR, 'testset-levela.tsv'), 
                        sep='\t', header=None, names=['id','tweet'])
test_lbl = pd.read_csv(os.path.join(DATA_DIR, 'labels-levela.csv'),
                        header=None, names=['id','label_str'], dtype={'id': str})
test_raw['id'] = test_raw['id'].astype(str)
df_test = test_raw.merge(test_lbl, on='id')
df_test['label'] = df_test['label_str'].map(LABEL2ID)
df_test = df_test.dropna(subset=['label'])
df_test['label'] = df_test['label'].astype(int)

# ── Reset index ──
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'✅ Dataset siap!')
print(f'   Training  : {len(df_train):,} baris')
print(f'   Validation: {len(df_val):,} baris')
print(f'   Test      : {len(df_test):,} baris')
print(f'\nDistribusi Training:')
print(df_train['label'].value_counts().rename({0:'NOT', 1:'OFF'}).to_string())

✅ Dataset siap!
   Training  : 11,916 baris
   Validation: 1,324 baris
   Test      : 860 baris

Distribusi Training:
label
NOT    7956
OFF    3960


---
## STEP 4 — Preprocessing Teks

**Mengapa perlu preprocessing?**  
Tweet mengandung banyak noise: URL, mention (@user), karakter berulang. Ini tidak informatif dan menghabiskan "slot" token yang terbatas (128 token).

**Apa yang dilakukan:**
- Hapus URL → tidak informatif
- Anonimkan @mention → jaga privasi + generalisasi
- Hapus simbol # (tapi simpan teks hashtag-nya)
- Normalisasi karakter berulang: `sooooo` → `so`

In [6]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)         # hapus URL
    text = re.sub(r'@\w+', '@USER', text)               # anonimkan mention
    text = re.sub(r'#(\w+)', r'\1', text)              # hapus simbol #
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)        # kurangi char berulang
    text = re.sub(r'\s+', ' ', text).strip()            # normalisasi spasi
    return text

# Terapkan ke semua dataset
df_train['text'] = df_train['tweet'].apply(clean_text)
df_val['text']   = df_val['tweet'].apply(clean_text)
df_test['text']  = df_test['tweet'].apply(clean_text)

# Contoh sebelum vs sesudah
print('Contoh preprocessing:')
for i in range(3):
    raw = df_train['tweet'].iloc[i]
    clean = df_train['text'].iloc[i]
    if raw != clean:
        print(f'  Sebelum: {raw}')
        print(f'  Sesudah: {clean}')
        print()

print('✅ Preprocessing selesai!')

Contoh preprocessing:
✅ Preprocessing selesai!


---
## STEP 5 — Tokenisasi & PyTorch Dataset

**Tokenisasi** = mengubah teks menjadi angka yang dimengerti model.

Setiap tweet diubah menjadi:
- `input_ids`: urutan angka yang merepresentasikan token
- `attention_mask`: 1 untuk token asli, 0 untuk padding

**PyTorch Dataset** = wrapper yang membungkus data agar bisa dibaca dalam batch.

In [7]:
print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('✅ Tokenizer siap!')

# Contoh tokenisasi
sample = df_train['text'].iloc[0]
enc = tokenizer(sample, max_length=MAX_LENGTH, truncation=True, padding='max_length')
print(f'\nContoh tokenisasi:')
print(f'  Teks          : {sample[:60]}...')
print(f'  Jumlah token  : {sum(1 for x in enc["attention_mask"] if x == 1)}')
print(f'  Token pertama : {tokenizer.convert_ids_to_tokens(enc["input_ids"][:8])}')

Loading tokenizer: bert-base-uncased ...


✅ Tokenizer siap!

Contoh tokenisasi:
  Teks          : @USER I think I would pick that just for the fun of it😂😂...
  Jumlah token  : 16
  Token pertama : ['[CLS]', '@', 'user', 'i', 'think', 'i', 'would', 'pick']


In [8]:
class OLIDDataset(Dataset):
    """
    PyTorch Dataset untuk OLID.
    __len__    → berapa jumlah data?
    __getitem__ → ambil satu data ke-idx
    """
    def __init__(self, df, tokenizer, max_length):
        self.texts     = df['text'].tolist()
        self.labels    = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long),
        }

# Buat dataset
train_ds = OLIDDataset(df_train, tokenizer, MAX_LENGTH)
val_ds   = OLIDDataset(df_val,   tokenizer, MAX_LENGTH)
test_ds  = OLIDDataset(df_test,  tokenizer, MAX_LENGTH)

# DataLoader: membaca data dalam batch
# num_workers=0 untuk Windows/Mac lokal (kalau error coba 0)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'✅ Dataset & DataLoader siap!')
print(f'   Train batches: {len(train_loader)}')
print(f'   Val batches  : {len(val_loader)}')
print(f'   Test batches : {len(test_loader)}')

✅ Dataset & DataLoader siap!
   Train batches: 373
   Val batches  : 42
   Test batches : 27


---
## STEP 6 — Load Model
 load model pre-trained dari HuggingFace, lalu tambahkan **classification head** (layer linear 768→2) di atasnya.

Fine-tuning = semua weight (pre-trained + classification head baru) diupdate saat training.

In [9]:
print(f'Loading model: {MODEL_NAME} ...')
print('(Proses ini akan download model ~400MB jika belum ada di cache)')

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
).to(device)

# Hitung jumlah parameter
total_params   = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n✅ Model siap!')
print(f'   Total parameter    : {total_params:,}')
print(f'   Trainable parameter: {trainable_params:,}')

Loading model: bert-base-uncased ...
(Proses ini akan download model ~400MB jika belum ada di cache)


Loading weights: 100%|██████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 9123.04it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initiali


✅ Model siap!
   Total parameter    : 109,483,778
   Trainable parameter: 109,483,778


---
## STEP 7 — Hitung Class Weights

**Mengapa weighted loss?**  
Karena data tidak seimbang (NOT 67% vs OFF 33%), model cenderung "malas" memprediksi OFF.  
Weighted loss memberi **penalti lebih besar** jika model salah memprediksi kelas minoritas.

**Formula:**  
`w_i = N_total / (C × N_i)`  
→ Kelas dengan data sedikit → N_i kecil → w_i besar → penalti lebih besar

In [10]:
def compute_weights(df, num_classes=2):
    counts = df['label'].value_counts().sort_index()
    n = len(df)
    weights = [n / (num_classes * counts.get(i, 1)) for i in range(num_classes)]
    return torch.tensor(weights, dtype=torch.float)

class_weights = compute_weights(df_train).to(device)

print(f'Bobot kelas:')
print(f'   NOT (kelas 0): {class_weights[0]:.4f}')
print(f'   OFF (kelas 1): {class_weights[1]:.4f}')
print(f'   → OFF dikalikan {class_weights[1]/class_weights[0]:.1f}x lebih besar dari NOT')

# Loss function
if USE_WEIGHTED:
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print(f'\n✅ Menggunakan WEIGHTED CrossEntropyLoss')
else:
    criterion = nn.CrossEntropyLoss()
    print(f'\n✅ Menggunakan CrossEntropyLoss TANPA bobot (ablasi)')

Bobot kelas:
   NOT (kelas 0): 0.7489
   OFF (kelas 1): 1.5045
   → OFF dikalikan 2.0x lebih besar dari NOT

✅ Menggunakan WEIGHTED CrossEntropyLoss


---
## STEP 8 — Training

**Optimizer AdamW** dengan learning rate rendah (2e-5) agar tidak merusak pre-trained weights.  
**Linear warmup** = LR naik perlahan 10% epoch pertama, lalu turun linear.  
**Gradient clipping** = cegah gradient meledak (exploding gradient).

In [11]:
# ── Setup optimizer & scheduler ──
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.1 * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Total training steps : {total_steps}')
print(f'Warmup steps         : {warmup_steps}')

def evaluate(model, loader):
    """Evaluasi model pada satu DataLoader."""
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0
    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(device)
            mask  = batch['attention_mask'].to(device)
            labels= batch['label'].to(device)
            logits= model(input_ids=ids, attention_mask=mask).logits
            loss  = criterion(logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return total_loss / len(loader), macro_f1, all_preds, all_labels

print('\n✅ Optimizer & scheduler siap. Mulai training...')

Total training steps : 1865
Warmup steps         : 186

✅ Optimizer & scheduler siap. Mulai training...


In [ ]:
# ══════════════════════════════════════
# TRAINING LOOP
# ══════════════════════════════════════
best_val_f1   = 0
best_epoch    = 0
best_preds    = None
best_labels   = None
history       = []

CKPT_DIR = os.path.join(OUTPUT_DIR, f'{RUN_NAME}_best')

for epoch in range(1, EPOCHS + 1):
    # ── Training ──
    model.train()
    train_loss = 0
    for step, batch in enumerate(train_loader):
        ids   = batch['input_ids'].to(device)
        mask  = batch['attention_mask'].to(device)
        labels= batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=mask).logits
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

        # Progress setiap 50 step
        if (step + 1) % 50 == 0:
            print(f'  Epoch {epoch} | Step {step+1}/{len(train_loader)} | loss={loss.item():.4f}', end='\r')

    avg_train_loss = train_loss / len(train_loader)

    # ── Validasi ──
    val_loss, val_f1, _, _ = evaluate(model, val_loader)

    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 
                    'val_loss': val_loss, 'val_f1': val_f1})

    print(f'Epoch {epoch}/{EPOCHS} | train_loss={avg_train_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1={val_f1:.4f}')

    # ── Simpan model terbaik ──
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        model.save_pretrained(CKPT_DIR)
        tokenizer.save_pretrained(CKPT_DIR)
        print(f'  ✅ Model terbaik disimpan (val_f1={best_val_f1:.4f})')

print(f'\n🏁 Training selesai! Best epoch: {best_epoch} | Best val_f1: {best_val_f1:.4f}')

  Epoch 1 | Step 50/373 | loss=0.6896

In [ ]:
# Visualisasi training curve
df_hist = pd.DataFrame(history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'Training Curve — {RUN_NAME}', fontweight='bold')

# Loss
axes[0].plot(df_hist['epoch'], df_hist['train_loss'], 'b-o', label='Train Loss')
axes[0].plot(df_hist['epoch'], df_hist['val_loss'],   'r-o', label='Val Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss per Epoch'); axes[0].legend()
axes[0].set_xticks(df_hist['epoch'])

# F1
axes[1].plot(df_hist['epoch'], df_hist['val_f1'], 'g-o', label='Val Macro F1')
axes[1].axvline(best_epoch, color='orange', linestyle='--', label=f'Best epoch={best_epoch}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Macro F1')
axes[1].set_title('Macro F1 per Epoch'); axes[1].legend()
axes[1].set_xticks(df_hist['epoch'])

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'curve_{RUN_NAME}.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Grafik disimpan!')

---
## STEP 9 — Evaluasi pada Test Set

Load model terbaik (checkpoint), lalu evaluasi pada test set resmi OLID.

In [ ]:
# Load model terbaik
print(f'Loading best model dari: {CKPT_DIR}')
best_model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR).to(device)

# Evaluasi test set
_, test_f1, test_preds, test_labels = evaluate(best_model, test_loader)

print(f'\n{"="*45}')
print(f'HASIL TEST SET — {RUN_NAME.upper()}')
print(f'{"="*45}')
print(f'Test Macro F1  : {test_f1:.4f}')
print(f'Test Accuracy  : {accuracy_score(test_labels, test_preds):.4f}')
print(f'\nClassification Report:')
print(classification_report(test_labels, test_preds, 
                              target_names=['NOT', 'OFF'], digits=4))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['NOT', 'OFF'], yticklabels=['NOT', 'OFF'],
            ax=ax, linewidths=1)
ax.set_xlabel('Prediksi', fontsize=12)
ax.set_ylabel('Aktual', fontsize=12)
ax.set_title(f'Confusion Matrix — {RUN_NAME}', fontweight='bold')

# Tambahkan keterangan
tn, fp, fn, tp = cm.ravel()
print(f'\nDetail Confusion Matrix:')
print(f'  TN (NOT benar diprediksi NOT) : {tn}')
print(f'  FP (NOT salah diprediksi OFF) : {fp}')
print(f'  FN (OFF salah diprediksi NOT) : {fn}  ← ini yang penting dikecilkan')
print(f'  TP (OFF benar diprediksi OFF) : {tp}')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'cm_{RUN_NAME}.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## STEP 10 — Simpan Hasil

Simpan prediksi dan metrik agar bisa dipakai untuk McNemar's Test nanti.

In [ ]:
import json

# Simpan prediksi (untuk McNemar's test)
np.save(os.path.join(OUTPUT_DIR, f'preds_{RUN_NAME}.npy'), np.array(test_preds))
np.save(os.path.join(OUTPUT_DIR, f'labels_test_a.npy'),   np.array(test_labels))

# Simpan ringkasan hasil
result = {
    'run_name'       : RUN_NAME,
    'model'          : MODEL_NAME,
    'seed'           : SEED,
    'weighted_loss'  : USE_WEIGHTED,
    'best_epoch'     : best_epoch,
    'best_val_f1'    : round(best_val_f1, 4),
    'test_macro_f1'  : round(test_f1, 4),
    'test_accuracy'  : round(accuracy_score(test_labels, test_preds), 4),
}

result_path = os.path.join(OUTPUT_DIR, f'result_{RUN_NAME}.json')
with open(result_path, 'w') as f:
    json.dump(result, f, indent=2)

print('✅ Hasil disimpan!')
print(f'   Prediksi : output/preds_{RUN_NAME}.npy')
print(f'   Hasil    : output/result_{RUN_NAME}.json')
print(f'   Checkpoint: output/{RUN_NAME}_best/')

print(f'\n{"="*45}')
print('RINGKASAN RUN INI')
print(f'{"="*45}')
for k, v in result.items():
    print(f'  {k:<20}: {v}')

print(f'\n➡️  Lanjut: jalankan ulang notebook ini dengan seed berbeda (123, 999)')
print(f'➡️  Setelah semua run selesai, jalankan: 03_mcnemar_test.ipynb')

---
## Ringkasan Cara Menjalankan Semua Run Task A

| Run | MODEL_CHOICE | SEED | USE_WEIGHTED |
|-----|-------------|------|-------------|
| 1   | `'bert'`    | `42` | `True`      |
| 2   | `'bert'`    | `123`| `True`      |
| 3   | `'bert'`    | `999`| `True`      |
| 4   | `'deberta'` | `42` | `True`      |
| 5   | `'deberta'` | `123`| `True`      |
| 6   | `'deberta'` | `999`| `True`      |
| 7   | `'bert'`    | `42` | `False`     |
| 8   | `'bert'`    | `123`| `False`     |
| 9   | `'bert'`    | `999`| `False`     |
| 10  | `'deberta'` | `42` | `False`     |
| 11  | `'deberta'` | `123`| `False`     |
| 12  | `'deberta'` | `999`| `False`     |

Ubah nilai di **STEP 2**, lalu `Kernel > Restart & Run All` untuk setiap baris di tabel.

---
*Kelompok 7 — NLP 2025/2026 | Amin Hardiansyah (D082251046) & Amalia Ramadhani (D082251049)*  
*Dosen: Dr. Muhammad Abdillah Rahmat, S.T., M.T.*